# Shape Mapper Group-Action Evaluation

Evaluate released shape mappers on every dumped toys4k mesh. Identity, inverse, composition, and six 60-degree rotations are generated deterministically from each shape id. Prediction-to-prediction comparisons measure feature cosine consistency. Prediction-to-GT comparisons additionally measure latent errors and GT-guided decoder output and subdivision agreement.

In [1]:
import hashlib
import math
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import trellis2.models as trellis2_models
from o_voxel.convert.flexible_dual_grid import intersect_occ, mesh_to_flexible_dual_grid
from pytorch3d.transforms import axis_angle_to_matrix, quaternion_to_matrix
from tqdm.auto import tqdm
from trellis2.modules.sparse import SparseTensor

from symtrellis.geometry import t_abs2grid
from symtrellis.mapper import from_pretrained

torch.set_grad_enabled(False)

MAPPER_NAMES = [
    "trellis2/shape/neighbor_graph/finetune",
    "trellis2/shape/swin3d/legacy",
]

DUMPED_MESH_DIR = Path("/mnt/scratch/trellis500k/toys4k/trellis2/dumped_mesh")
SHAPE_ENCODER_ID = "microsoft/TRELLIS.2-4B/ckpts/shape_enc_next_dc_f16c32_fp16"
SHAPE_DECODER_ID = "microsoft/TRELLIS.2-4B/ckpts/shape_dec_next_dc_f16c32_fp16"

DEVICE = torch.device("cuda:0")
GLOBAL_SEED = 20260701
NUM_RANDOM_TRIALS = 2
CLOSURE_STEPS = 6
SHAPE_EVAL_BATCH_SIZE = 4
LATENT_GRID_SIZE = 32
OVOXEL_GRID_SIZE = 512
OVOXEL_TO_LATENT = OVOXEL_GRID_SIZE // LATENT_GRID_SIZE
DECODE_RESOLUTION = 512
AABB = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]]

[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn


In [2]:
mesh_paths = sorted(DUMPED_MESH_DIR.glob('*.pickle'))
print(f'dumped meshes: {len(mesh_paths):,}')

shape_encoder = trellis2_models.from_pretrained(SHAPE_ENCODER_ID).eval().to(DEVICE)
shape_decoder = trellis2_models.from_pretrained(SHAPE_DECODER_ID)
shape_decoder.convert_to_fp32()
shape_decoder.use_fp16 = False
shape_decoder.dtype = torch.float32
shape_decoder.set_resolution(DECODE_RESOLUTION)
shape_decoder.float().eval().to(DEVICE)

for parameter in shape_encoder.parameters():
    parameter.requires_grad_(False)
for parameter in shape_decoder.parameters():
    parameter.requires_grad_(False)


dumped meshes: 3,229


In [3]:
def rotation_for_trial(shape_id, operation, trial_id, rotation_id):
    key = f'{GLOBAL_SEED}:{shape_id}:{operation}:{trial_id}:{rotation_id}'
    seed = int.from_bytes(hashlib.sha256(key.encode('ascii')).digest()[:8], 'big')
    generator = torch.Generator(device='cpu')
    generator.manual_seed(seed)

    if operation == 'closure':
        axis = torch.randn(3, generator=generator)
        axis = axis / axis.norm().clamp_min(1e-12)
        angle = 2.0 * math.pi / CLOSURE_STEPS
        return axis_angle_to_matrix((axis * angle)[None])[0].to(DEVICE)

    quaternion = torch.randn(4, generator=generator)
    quaternion = quaternion / quaternion.norm().clamp_min(1e-12)
    return quaternion_to_matrix(quaternion[None])[0].to(DEVICE)


@torch.no_grad()
def voxelize_shape_mesh(meshes, rotations):
    voxel_states = []
    support_coords = []
    for sample_id, (mesh, rotation) in enumerate(zip(meshes, rotations)):
        vertices, faces = mesh
        rotated_vertices = (vertices @ rotation.T).contiguous()
        occupied_coords = intersect_occ(
            vertices=rotated_vertices,
            faces=faces,
            grid_size=OVOXEL_GRID_SIZE,
            aabb=AABB,
        )
        latent_xyz = torch.unique(occupied_coords.to(dtype=torch.int32) // OVOXEL_TO_LATENT, dim=0)
        sort_key = (
            latent_xyz[:, 0].long() * LATENT_GRID_SIZE * LATENT_GRID_SIZE
            + latent_xyz[:, 1].long() * LATENT_GRID_SIZE
            + latent_xyz[:, 2].long()
        )
        latent_xyz = latent_xyz[torch.argsort(sort_key)]
        batch_column = torch.full(
            (latent_xyz.shape[0], 1), sample_id, device=DEVICE, dtype=torch.int32
        )
        support_coords.append(torch.cat([batch_column, latent_xyz], dim=1))
        voxel_states.append((rotated_vertices, faces))
    return voxel_states, torch.cat(support_coords, dim=0).contiguous()


@torch.no_grad()
def encode_shape_mesh(voxel_states, encoder):
    dual_coords = []
    dual_offsets = []
    intersections = []
    for sample_id, (vertices, faces) in enumerate(voxel_states):
        voxel_coords, dual_vertices, intersected = mesh_to_flexible_dual_grid(
            vertices=vertices,
            faces=faces,
            grid_size=OVOXEL_GRID_SIZE,
            aabb=AABB,
            face_weight=1.0,
            boundary_weight=0.2,
            regularization_weight=1e-2,
            timing=False,
        )
        offsets = (dual_vertices * OVOXEL_GRID_SIZE - voxel_coords).clamp(0.0, 1.0).float()
        batch_column = torch.full(
            (voxel_coords.shape[0], 1), sample_id, device=DEVICE, dtype=torch.int32
        )
        dual_coords.append(torch.cat([batch_column, voxel_coords.to(dtype=torch.int32)], dim=1))
        dual_offsets.append(offsets)
        intersections.append(intersected.to(dtype=torch.bool))

    coords = torch.cat(dual_coords, dim=0).contiguous()
    vertices_sp = SparseTensor(feats=torch.cat(dual_offsets, dim=0), coords=coords)
    intersected_sp = SparseTensor(feats=torch.cat(intersections, dim=0), coords=coords)
    latent = encoder(vertices_sp, intersected_sp)
    return latent.coords.to(dtype=torch.int32), latent.feats.float(), len(voxel_states)


@torch.no_grad()
def apply_shape_mapper(model, source_state, destination_state, source_pose, destination_pose):
    coords_src, feats_src, batch_size = source_state
    coords_dst = destination_state[0]
    # Mapper transforms are destination-to-source: R_src @ R_dst.T.
    O_dst2src = torch.bmm(source_pose, destination_pose.transpose(1, 2)).float()
    zero_translation = torch.zeros((batch_size, 3), device=DEVICE, dtype=torch.float32)
    coeff = model(
        coords_src=coords_src,
        coords_dst=coords_dst,
        O_dst2src=O_dst2src,
        t_dst2src=t_abs2grid(zero_translation, O_dst2src, LATENT_GRID_SIZE),
        s_dst2src=torch.ones(batch_size, device=DEVICE, dtype=torch.long),
    ).to(device=DEVICE, dtype=torch.float32)
    mapped = coeff.apply(feats_src.to(dtype=coeff.dtype)).float()
    destination_has_edge = torch.bincount(coeff.e_ids_dst, minlength=coeff.num_dst) > 0
    mapped = mapped * destination_has_edge[:, None]
    return coords_dst, mapped, batch_size


def decode_shape_features(decoder, latent, guide_subdivisions=None):
    h = decoder.from_latent(latent)
    h = h.type(decoder.dtype)
    subdivision_logits = []

    for stage_index, blocks in enumerate(decoder.blocks):
        for block_index, block in enumerate(blocks):
            is_subdivision_block = (
                stage_index < len(decoder.blocks) - 1
                and block_index == len(blocks) - 1
            )
            if not is_subdivision_block:
                h = block(h)
                continue

            if guide_subdivisions is None:
                h, subdivision = block(h)
                subdivision_logits.append(subdivision)
                continue

            subdivision = block.to_subdiv(h)
            # Reuse the GT subdivision lattice so every prediction is compared on identical coordinates.
            guide = guide_subdivisions[len(subdivision_logits)]
            guide_mask = guide.replace(guide.feats > 0)
            residual = h
            h = h.replace(block.norm1(h.feats))
            h = h.replace(F.silu(h.feats))
            h = block.conv1(h)
            h = block.updown(h, guide_mask)
            residual = block.updown(residual, guide_mask)
            h = h.replace(block.norm2(h.feats))
            h = h.replace(F.silu(h.feats))
            h = block.conv2(h)
            h = h + block.skip_connection(residual)
            subdivision_logits.append(subdivision)

    h = h.type(latent.dtype)
    h = h.replace(F.layer_norm(h.feats, h.feats.shape[-1:]))
    return decoder.output_layer(h), subdivision_logits


In [4]:
def compute_shape_consistency_metrics(lhs, rhs):
    coords, lhs_features, batch_size = lhs
    rhs_features = rhs[1]
    cosine_values = []
    for sample_id in range(batch_size):
        sample_mask = coords[:, 0].long() == sample_id
        cosine_values.append(
            F.cosine_similarity(
                lhs_features[sample_mask].float(),
                rhs_features[sample_mask].float(),
                dim=1,
            ).mean()
        )
    return {'feature_cosine': torch.stack(cosine_values)}


@torch.no_grad()
def compute_shape_gt_metrics(prediction, target, target_output, target_subdivisions, decoder):
    coords, prediction_features, batch_size = prediction
    target_features = target[1]
    decoder_dtype = next(decoder.parameters()).dtype
    prediction_slat = SparseTensor(
        feats=prediction_features.to(dtype=decoder_dtype),
        coords=coords,
    )
    prediction_output, prediction_subdivisions = decode_shape_features(
        decoder, prediction_slat, target_subdivisions
    )

    metric_values = {
        'feature_l1': [],
        'feature_l2': [],
        'feature_cosine': [],
        'feature_cosine_distance': [],
        'decoder_output_l2': [],
    }
    for stage_index in range(len(target_subdivisions)):
        metric_values[f'decoder_subdivision_l2_stage_{stage_index}'] = []
        metric_values[f'decoder_subdivision_iou_stage_{stage_index}'] = []

    output_batch_ids = target_output.coords[:, 0].long()
    subdivision_batch_ids = [subdivision.coords[:, 0].long() for subdivision in target_subdivisions]
    for sample_id in range(batch_size):
        feature_mask = coords[:, 0].long() == sample_id
        sample_prediction = prediction_features[feature_mask].float()
        sample_target = target_features[feature_mask].float()
        feature_difference = sample_prediction - sample_target
        feature_cosine = F.cosine_similarity(sample_prediction, sample_target, dim=1).mean()
        metric_values['feature_l1'].append(feature_difference.abs().mean(dim=1).mean())
        metric_values['feature_l2'].append(feature_difference.square().mean(dim=1).mean())
        metric_values['feature_cosine'].append(feature_cosine)
        metric_values['feature_cosine_distance'].append(1.0 - feature_cosine)

        output_mask = output_batch_ids == sample_id
        output_difference = (
            prediction_output.feats[output_mask].float()
            - target_output.feats[output_mask].float()
        )
        metric_values['decoder_output_l2'].append(output_difference.square().mean())

        for stage_index, (prediction_subdivision, target_subdivision) in enumerate(
            zip(prediction_subdivisions, target_subdivisions)
        ):
            subdivision_mask = subdivision_batch_ids[stage_index] == sample_id
            prediction_logits = prediction_subdivision.feats[subdivision_mask].float()
            target_logits = target_subdivision.feats[subdivision_mask].float()
            metric_values[f'decoder_subdivision_l2_stage_{stage_index}'].append(
                (prediction_logits - target_logits).square().mean()
            )
            prediction_occupied = prediction_logits > 0
            target_occupied = target_logits > 0
            intersection = (prediction_occupied & target_occupied).sum().float()
            union = (prediction_occupied | target_occupied).sum().float()
            metric_values[f'decoder_subdivision_iou_stage_{stage_index}'].append(
                intersection / union.clamp_min(1.0)
            )

    return {name: torch.stack(values) for name, values in metric_values.items()}


def accumulate_metrics(metric_sums, metric_counts, prefix, metrics):
    for name, values in metrics.items():
        key = f'{prefix}_{name}'
        metric_sums[key] = metric_sums.get(
            key, values.new_zeros((), dtype=torch.float64)
        ) + values.double().sum()
        metric_counts[key] = metric_counts.get(key, 0) + values.shape[0]


In [5]:
@torch.no_grad()
def evaluate_shape_group_actions(mesh_paths, model, encoder, decoder):
    metric_sums = {}
    metric_counts = {}
    model.eval()
    decoder_dtype = next(decoder.parameters()).dtype

    for batch_start in tqdm(
        range(0, len(mesh_paths), SHAPE_EVAL_BATCH_SIZE),
        desc='evaluate shape',
        leave=False,
    ):
        batch_paths = mesh_paths[batch_start:batch_start + SHAPE_EVAL_BATCH_SIZE]
        shape_ids = [path.stem for path in batch_paths]
        meshes = []
        for path in batch_paths:
            with path.open('rb') as file:
                dump = pickle.load(file)
            vertex_arrays = []
            face_arrays = []
            vertex_offset = 0
            for obj in dump['objects']:
                if obj['vertices'].size == 0 or obj['faces'].size == 0:
                    continue
                vertex_arrays.append(obj['vertices'])
                face_arrays.append(obj['faces'] + vertex_offset)
                vertex_offset += len(obj['vertices'])
            vertices = torch.from_numpy(np.concatenate(vertex_arrays, axis=0)).to(DEVICE, dtype=torch.float32)
            faces = torch.from_numpy(np.concatenate(face_arrays, axis=0)).to(DEVICE, dtype=torch.long)
            meshes.append((vertices.contiguous(), faces.contiguous()))

        batch_size = len(meshes)
        identity_pose = torch.eye(3, device=DEVICE).expand(batch_size, -1, -1).clone()
        identity_voxels, _ = voxelize_shape_mesh(meshes, identity_pose)
        identity_gt = encode_shape_mesh(identity_voxels, encoder)
        identity_target_slat = SparseTensor(
            feats=identity_gt[1].to(dtype=decoder_dtype),
            coords=identity_gt[0],
        )
        identity_target_output, identity_target_subdivisions = decode_shape_features(
            decoder, identity_target_slat
        )
        identity_prediction = apply_shape_mapper(
            model, identity_gt, identity_gt, identity_pose, identity_pose
        )
        accumulate_metrics(
            metric_sums,
            metric_counts,
            'identity_gt',
            compute_shape_gt_metrics(
                identity_prediction,
                identity_gt,
                identity_target_output,
                identity_target_subdivisions,
                decoder,
            ),
        )

        for trial_id in range(NUM_RANDOM_TRIALS):
            inverse_pose = torch.stack(
                [rotation_for_trial(shape_id, 'inverse', trial_id, 0) for shape_id in shape_ids]
            )
            _, inverse_support = voxelize_shape_mesh(meshes, inverse_pose)
            inverse_destination = (inverse_support, None, batch_size)
            inverse_forward = apply_shape_mapper(
                model, identity_gt, inverse_destination, identity_pose, inverse_pose
            )
            inverse_prediction = apply_shape_mapper(
                model, inverse_forward, identity_gt, inverse_pose, identity_pose
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'inverse_consistency',
                compute_shape_consistency_metrics(inverse_prediction, identity_prediction),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'inverse_gt',
                compute_shape_gt_metrics(
                    inverse_prediction,
                    identity_gt,
                    identity_target_output,
                    identity_target_subdivisions,
                    decoder,
                ),
            )

            first_pose = torch.stack(
                [rotation_for_trial(shape_id, 'composition', trial_id, 0) for shape_id in shape_ids]
            )
            second_rotation = torch.stack(
                [rotation_for_trial(shape_id, 'composition', trial_id, 1) for shape_id in shape_ids]
            )
            final_pose = torch.bmm(second_rotation, first_pose)
            _, first_support = voxelize_shape_mesh(meshes, first_pose)
            final_voxels, _ = voxelize_shape_mesh(meshes, final_pose)
            first_destination = (first_support, None, batch_size)
            composition_gt = encode_shape_mesh(final_voxels, encoder)
            composition_target_slat = SparseTensor(
                feats=composition_gt[1].to(dtype=decoder_dtype),
                coords=composition_gt[0],
            )
            composition_target_output, composition_target_subdivisions = decode_shape_features(
                decoder, composition_target_slat
            )
            first_prediction = apply_shape_mapper(
                model, identity_gt, first_destination, identity_pose, first_pose
            )
            sequential_prediction = apply_shape_mapper(
                model, first_prediction, composition_gt, first_pose, final_pose
            )
            direct_prediction = apply_shape_mapper(
                model, identity_gt, composition_gt, identity_pose, final_pose
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'composition_consistency',
                compute_shape_consistency_metrics(sequential_prediction, direct_prediction),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'composition_sequential_gt',
                compute_shape_gt_metrics(
                    sequential_prediction,
                    composition_gt,
                    composition_target_output,
                    composition_target_subdivisions,
                    decoder,
                ),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'composition_direct_gt',
                compute_shape_gt_metrics(
                    direct_prediction,
                    composition_gt,
                    composition_target_output,
                    composition_target_subdivisions,
                    decoder,
                ),
            )

            closure_rotation = torch.stack(
                [rotation_for_trial(shape_id, 'closure', trial_id, 0) for shape_id in shape_ids]
            )
            closure_prediction = identity_gt
            source_pose = identity_pose
            for closure_step in range(CLOSURE_STEPS):
                destination_pose = torch.bmm(closure_rotation, source_pose)
                if closure_step == CLOSURE_STEPS - 1:
                    destination_pose = identity_pose
                    destination_state = identity_gt
                else:
                    _, closure_support = voxelize_shape_mesh(meshes, destination_pose)
                    destination_state = (closure_support, None, batch_size)
                closure_prediction = apply_shape_mapper(
                    model, closure_prediction, destination_state, source_pose, destination_pose
                )
                source_pose = destination_pose
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'closure_consistency',
                compute_shape_consistency_metrics(closure_prediction, identity_prediction),
            )
            accumulate_metrics(
                metric_sums,
                metric_counts,
                'closure_gt',
                compute_shape_gt_metrics(
                    closure_prediction,
                    identity_gt,
                    identity_target_output,
                    identity_target_subdivisions,
                    decoder,
                ),
            )

    return {name: (value / metric_counts[name]).item() for name, value in metric_sums.items()}


In [6]:
records = []
for model_name in tqdm(MAPPER_NAMES, desc="models"):
    print(f"Evaluating {model_name}")
    mapper = from_pretrained(model_name, device=DEVICE).eval()
    metrics = evaluate_shape_group_actions(mesh_paths, mapper, shape_encoder, shape_decoder)
    records.append({"model_name": model_name, **metrics})
    del mapper
    torch.cuda.empty_cache()

models:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating trellis2/shape/neighbor_graph/finetune


evaluate shape:   0%|          | 0/808 [00:00<?, ?it/s]

Evaluating trellis2/shape/swin3d/legacy


evaluate shape:   0%|          | 0/808 [00:00<?, ?it/s]

In [7]:
evaluation_df = pd.DataFrame(records)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
evaluation_df.set_index("model_name").T

model_name,trellis2/shape/neighbor_graph/finetune,trellis2/shape/swin3d/legacy
identity_gt_feature_l1,0.855273,0.932805
identity_gt_feature_l2,1.244266,1.463770
identity_gt_feature_cosine,0.981070,0.980316
identity_gt_feature_cosine_distance,0.018930,0.019684
identity_gt_decoder_output_l2,2.291293,4.460121
identity_gt_decoder_subdivision_l2_stage_0,158.405829,479.398642
identity_gt_decoder_subdivision_iou_stage_0,0.994452,0.972060
identity_gt_decoder_subdivision_l2_stage_1,206.069494,379.968675
identity_gt_decoder_subdivision_iou_stage_1,0.977387,0.958625
identity_gt_decoder_subdivision_l2_stage_2,130.203892,289.610495
